# Fase 0: Carregamento de Dados - Clustering de Olhos Barrett II

## Objetivo
Carregar e limpar os dados de clustering de olhos Barrett II.
- Dados biometricos: AL, ACD, WTW, K1, K2
- Formato portugues (decimals com virgula)
- Variavel Correto (S/N) nao faz parte do clustering

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os

# Configuracao
DATA_PATH = '/home/antonio/Documentos/Machine Learning/atividade_4/RTVue_20221110_MLClass.csv'
OUTPUT_DIR = '/home/antonio/Documentos/Machine Learning/atividade_4'

# Criar diretorios se necessario
os.makedirs(f'{OUTPUT_DIR}/imgs', exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/results', exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/notebooks', exist_ok=True)

print('Diretorios criados com sucesso!')

Diretorios criados com sucesso!


## Carregar Dados

In [2]:
# Carregar CSV com formato portugues (virgula como decimal)
df = pd.read_csv(DATA_PATH, decimal=',')

print(f'Shape: {df.shape}')
print(f'\nColunas: {df.columns.tolist()}')
print(f'\nPrimeiras 5 linhas:')
df.head()

Shape: (5972, 14)

Colunas: ['Index', 'pID', 'Age', 'Gender', 'Eye', 'C', 'S', 'ST', 'T', 'IT', 'I', 'IN', 'N', 'SN']

Primeiras 5 linhas:


,Index,pID,Age,Gender,Eye,C,S,ST,T,IT,I,IN,N,SN
0,0,6269,22,F,OS,50.0,56.0,60.0,71.0,57.0,54.0,52.0,52.0,51.0
1,2,6270,33,F,OS,56.0,55.0,54.0,54.0,55.0,56.0,56.0,55.0,55.0
2,4,6274,34,F,OS,58.0,69.0,55.0,57.0,67.0,72.0,66.0,59.0,64.0
3,7,6277,12,M,OS,47.0,52.0,52.0,47.0,46.0,48.0,55.0,56.0,53.0
4,9,6278,25,M,OS,53.0,52.0,51.0,52.0,54.0,55.0,54.0,53.0,53.0


## Inspecao do DataFrame

In [3]:
# Verificar tipos de dados
print('Tipos de dados:')
print(df.dtypes)
print('\n--- Info Basica ---')
df.info()fdfdf

Tipos de dados:
Index       int64
pID         int64
Age         int64
Gender        str
Eye           str
C         float64
S         float64
ST        float64
T         float64
IT        float64
I         float64
IN        float64
N         float64
SN        float64
dtype: object

--- Info Basica ---
<class 'pandas.DataFrame'>
RangeIndex: 5972 entries, 0 to 5971
Data columns (total 14 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Index   5972 non-null   int64  
 1   pID     5972 non-null   int64  
 2   Age     5972 non-null   int64  
 3   Gender  5972 non-null   str    
 4   Eye     5972 non-null   str    
 5   C       5658 non-null   float64
 6   S       5850 non-null   float64
 7   ST      5841 non-null   float64
 8   T       5831 non-null   float64
 9   IT      5829 non-null   float64
 10  I       5807 non-null   float64
 11  IN      5838 non-null   float64
 12  N       5844 non-null   float64
 13  SN      5850 non-null   float64
dtypes: f

In [4]:
# Estatisticas basicas
print('Estatisticas basicas:')
df.describe()

Estatisticas basicas:


,Index,pID,Age,C,S,ST,T,IT,I,IN,N,SN
count,5972.000000,5972.000000,5972.000000,5658.000000,5850.000000,5841.000000,5831.000000,5829.000000,5807.000000,5838.000000,5844.000000,5850.000000
mean,3256.175988,3243.087073,39.283992,53.005479,55.028205,52.701079,52.283656,53.673357,54.221457,54.256423,54.141513,53.718803
std,1856.741695,2421.501471,40.511803,19.869972,134.803161,16.773767,16.823118,65.371721,24.090966,20.568430,33.434342,51.244372
min,0.000000,82.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1682.750000,832.500000,25.000000,49.000000,49.000000,49.000000,48.000000,49.000000,50.000000,51.000000,50.000000,50.000000
50%,3298.500000,4099.000000,34.000000,53.000000,52.000000,52.000000,52.000000,52.000000,54.000000,54.000000,54.000000,53.000000
75%,4853.250000,5507.000000,50.000000,56.000000,57.000000,56.000000,56.000000,57.000000,58.000000,58.000000,57.000000,57.000000
max,6424.000000,6956.000000,2019.000000,770.000000,7318.000000,739.000000,776.000000,4848.000000,849.000000,893.000000,2310.000000,3801.000000


In [5]:
# Verificar valores nulos
print('Valores nulos por coluna:')
print(df.isnull().sum())
print(f'\nTotal de linhas com valores nulos: {df.isnull().any(axis=1).sum()}')

Valores nulos por coluna:
Index       0
pID         0
Age         0
Gender      0
Eye         0
C         314
S         122
ST        131
T         141
IT        143
I         165
IN        134
N         128
SN        122
dtype: int64

Total de linhas com valores nulos: 685


## Limpeza de Dados

In [7]:
# Identificar colunas numericas (para clustering)
numeric_cols = ['Index', 'pID', 'Age', 'C', 'S', 'ST', 'T', 'IT', 'I', 'IN', 'N', 'SN']

# Verificar se todas existem
print('Colunas numericas para clustering:', numeric_cols)
print('Colunas no DataFrame:', df.columns.tolist())
print('\nColunas numeric_cols presentes:', all(col in df.columns for col in numeric_cols))

Colunas numericas para clustering: ['Index', 'pID', 'Age', 'C', 'S', 'ST', 'T', 'IT', 'I', 'IN', 'N', 'SN']
Colunas no DataFrame: ['Index', 'pID', 'Age', 'Gender', 'Eye', 'C', 'S', 'ST', 'T', 'IT', 'I', 'IN', 'N', 'SN']

Colunas numeric_cols presentes: True


In [9]:
# Criar dataset limpo para clustering
df_clean = df.copy()

# Garantir que colunas numericas sao float
for col in numeric_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

print(f'Dataset limpo - Shape: {df_clean.shape}')
print(f'\nTipos de dados pos conversao:')
print(df_clean[numeric_cols].dtypes)

Dataset limpo - Shape: (5972, 14)

Tipos de dados pos conversao:
Index      int64
pID        int64
Age        int64
C        float64
S        float64
ST       float64
T        float64
IT       float64
I        float64
IN       float64
N        float64
SN       float64
dtype: object


In [10]:
# Verificar valores nulos nas colunas numericas
print('Valores nulos pos conversao:')
print(df_clean[numeric_cols].isnull().sum())

# Remover linhas com valores nulos
df_clean = df_clean.dropna(subset=numeric_cols)
print(f'\nShape apos remover nulos: {df_clean.shape}')

Valores nulos pos conversao:
Index      0
pID        0
Age        0
C        314
S        122
ST       131
T        141
IT       143
I        165
IN       134
N        128
SN       122
dtype: int64

Shape apos remover nulos: (5287, 14)


## Salvar Dataset Limpo

In [11]:
# Salvar como CSV
df_clean.to_csv(f'{OUTPUT_DIR}/df_clean.csv', index=False)

print('Arquivos salvos:')
print(f'  - {OUTPUT_DIR}/df_clean.csv')
print(f'\nDataset limpo: {df_clean.shape[0]} registros, {len(numeric_cols)} features')

Arquivos salvos:
  - /home/antonio/Documentos/Machine Learning/atividade_4/df_clean.csv

Dataset limpo: 5287 registros, 12 features


## Resumo da Fase 0
- Dados carregados com formato portugues (virgula como decimal)
- 5 features biometricas: AL, ACD, WTW, K1, K2
- Variavel Correto (S/N) mantida para analise complementar
- Dataset limpo salvo para proximas fases